# PW02 · Source Merge And Global Thresholds

用途：在 Colab 冷启动环境中合并 PW01 双 role shard，构建 content 与 attestation 两条全局阈值链，并回读 PW02 summary。

## 1) 定义路径与参数

本单元只做常量定义与最小工具函数准备，不执行任何外部副作用。

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW02_Source_Merge_And_Global_Thresholds"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW02_Source_Merge_And_Global_Thresholds.py"

FAMILY_ID = "paper_eval_family_demo"

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

## 2) 挂载 Drive 并同步仓库

本单元保证 PW02 可以在空白 Colab 会话中直接冷启动。

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
    },
)

## 3) 运行前 precheck

本单元检查 PW02 依赖的 PW00/PW01 真源产物是否齐全。

In [ ]:
FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
FAMILY_MANIFEST_PATH = FAMILY_ROOT / "manifests" / "paper_eval_family_manifest.json"
SOURCE_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_shard_plan.json"
SOURCE_SPLIT_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_split_plan.json"
PW02_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw02_summary.json"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("Drive 项目根目录存在", DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW02 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("family 根目录存在", FAMILY_ROOT.exists(), str(FAMILY_ROOT))
record_precheck("family manifest 存在", FAMILY_MANIFEST_PATH.exists(), str(FAMILY_MANIFEST_PATH))
record_precheck("source shard plan 存在", SOURCE_SHARD_PLAN_PATH.exists(), str(SOURCE_SHARD_PLAN_PATH))
record_precheck("source split plan 存在", SOURCE_SPLIT_PLAN_PATH.exists(), str(SOURCE_SPLIT_PLAN_PATH))
record_precheck("positive shard 根目录存在", (FAMILY_ROOT / "source_shards" / "positive").exists(), str(FAMILY_ROOT / "source_shards" / "positive"))
record_precheck("negative shard 根目录存在", (FAMILY_ROOT / "source_shards" / "negative").exists(), str(FAMILY_ROOT / "source_shards" / "negative"))

print_json("pw02_precheck", {"items": PRECHECK_RESULTS})

hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW02 precheck failed: {hard_fail}")

## 4) 执行 PW02 CLI

本单元调用 paper_workflow/scripts/PW02_Source_Merge_And_Global_Thresholds.py，并回读标准 JSON summary。

In [ ]:
from scripts.notebook_runtime_common import build_repo_import_subprocess_env

COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
]

PW02_RESULT = subprocess.run(
    COMMAND,
    cwd=REPO_ROOT,
    env=build_repo_import_subprocess_env(repo_root=REPO_ROOT),
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if PW02_RESULT.returncode != 0:
    raise RuntimeError(
        "PW02 failed: "
        f"return_code={PW02_RESULT.returncode} stdout={PW02_RESULT.stdout} stderr={PW02_RESULT.stderr}"
    )

if not PW02_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "PW02 summary file missing after successful subprocess execution: "
        f"summary_path={PW02_SUMMARY_PATH} stdout={PW02_RESULT.stdout} stderr={PW02_RESULT.stderr}"
    )

PW02_SUMMARY = json.loads(PW02_SUMMARY_PATH.read_text(encoding="utf-8"))
print_json("pw02_summary", PW02_SUMMARY)